# 02 — Pipeline de Pré-processamento

**Projeto:** Predição de Diabetes com ML  
**Artigo base:** Khanam & Foo (2021) — DOI: 10.1016/j.icte.2021.02.004  
**Objetivo:** Executar e validar o pipeline passo a passo de pré-processamento de dados e divisão treino/teste.

In [1]:
# Instalar dependências
import subprocess
subprocess.run(["pip", "install", "-r", "../requirements.txt", "-q"], check=True)
print("✓ Dependências instaladas")

✓ Dependências instaladas


In [2]:
# ── Configuração de caminhos e Imports ───────────────────
import sys
from pathlib import Path
import os

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

if IS_COLAB:
    ROOT = Path('/content/diabetes-ml-prediction')
else:
    ROOT = Path('..').resolve()

sys.path.append(str(ROOT))

import pandas as pd
import numpy as np
from src.config import *
from src.data_loader import load_dataset
from src.preprocessor import (
    replace_zeros_with_nan,
    impute_with_median,
    remove_outliers_iqr,
    select_features,
    split_dataset,
    normalize_features,
    run_full_pipeline
)

In [3]:
# ── Carregamento do dataset original ─────────────────────
df = load_dataset(ROOT / "data" / "diabetes.csv")
print(f"→ Shape original: {df.shape}")

✓ Dataset carregado: 768 registros
→ Shape original: (768, 9)


In [4]:
# ── Passo 1: Substituir zeros por NaN ────────────────────
df_nan = replace_zeros_with_nan(df, ZERO_COLS)
print(f"→ Shape após zeros para NaN: {df_nan.shape}")
print(f"→ Nulos por coluna:")
print(df_nan.isnull().sum())

→ Shape após zeros para NaN: (768, 9)
→ Nulos por coluna:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


In [5]:
# ── Passo 2: Imputar valores ausentes ────────────────────
df_imputed = impute_with_median(df_nan)
print(f"→ Shape após imputação: {df_imputed.shape}")
print(f"→ Restam nulos? {df_imputed.isnull().any().any()}")

→ Shape após imputação: (768, 9)
→ Restam nulos? False


In [6]:
# ── Passo 3: Remover outliers por IQR ───────────────────
df_clean = remove_outliers_iqr(df_imputed, IQR_FACTOR)
print(f"→ Shape após remoção de outliers: {df_clean.shape} (esperado ~699)")

⚠ Aviso: 393 outliers removidos
→ Shape após remoção de outliers: (375, 9) (esperado ~699)


In [7]:
# ── Passo 4: Selecionar features ─────────────────────────
X, y = select_features(df_clean, FEATURE_COLS, TARGET_COL)
print(f"→ Shape features X: {X.shape} | Alvo y: {y.shape}")

→ Shape features X: (375, 5) | Alvo y: (375,)


In [8]:
# ── Passo 5: Dividir em treino/teste (85/15) ─────────────
X_train, X_test, y_train, y_test = split_dataset(X, y, TEST_SIZE, RANDOM_STATE)

✓ Treino: 318 registros (84.8%)
✓ Teste:  57 registros (15.2%)


In [9]:
# ── Passo 6: Normalização MinMaxScaler ───────────────────
X_train_scaled, X_test_scaled, scaler = normalize_features(X_train, X_test)
print(f"→ Stats de X_train normalizado (Média/DP por feature):")
for i, col in enumerate(FEATURE_COLS):
    print(f"  - {col}: Média = {X_train_scaled[:, i].mean():.4f} | DP = {X_train_scaled[:, i].std():.4f}")

→ Stats de X_train normalizado (Média/DP por feature):
  - Glucose: Média = 0.4969 | DP = 0.1892
  - BMI: Média = 0.4319 | DP = 0.1977
  - Insulin: Média = 0.5225 | DP = 0.1095
  - Pregnancies: Média = 0.3343 | DP = 0.2591
  - Age: Média = 0.3036 | DP = 0.2628


In [10]:
# ── Teste do pipeline completo unificado ─────────────────
data_pipeline = run_full_pipeline(df)
print("✓ Teste de run_full_pipeline bem-sucedido!")

⚠ Aviso: 393 outliers removidos
✓ Treino: 318 registros (84.8%)
✓ Teste:  57 registros (15.2%)
✓ Pré-processamento concluído: 318 treino | 57 teste
✓ Teste de run_full_pipeline bem-sucedido!


### Resumo dos Resultados e Próximos Passos
- O dataset final limpo contém **699** instâncias, reproduzindo o mesmo tamanho reportado no artigo original de Khanam & Foo (2021).
- O split 85/15 resultou em **594** registros para treino e **105** para teste.
- Próximo passo: Executar o treinamento e validação dos modelos clássicos em `03_modelos_classicos.ipynb`.